In [505]:
import chardet
from collections.abc import Mapping
from collections import OrderedDict
import chardet, csv, pandas as pd, io, os, textwrap
from datetime import datetime
import json
import pandas as pd
from pathlib import Path
import re
import sys
from typing import Any, List


### CAUSALITY Project Noteboook Mark Two

A notebook to ingest vuln data, add rating labels, and calculate stats on ratings and watchlist coverage. 
Resale and /or incorporation into a paid product or service is not covered by license.   
See https://github.com/opendr-io/causality/blob/main/LICENSE.md for details.  

Usage: In a venv, install any missing modules in the cell above. Next define the input files below. The next cell will do a directory listing of the current path to help locate where the notebook is relative to the data files.

In [ ]:
print("Current directory is:")
print(Path.cwd())
print("Here is the current directory listing - make sure your data files are here:")
for p in sorted(Path(".").iterdir(), key=lambda x: (not x.is_dir(), x.name.lower())):
    tag = "<DIR>" if p.is_dir() else "     "
    print(f"{tag}  {p.name}")

### Configure these input file names: 

- vuln_path - your vuln data in csv format. Most of the export I see are in csv, let me know if you need a json ingestor. 

Use the .txt files for the next two which are tab delimited. Sometimes CSV gets messy:

- file2024 - the name of your ratings file for 2024 (from the causality project; start with this one: https://github.com/opendr-io/causality/blob/main/2024/reduction.txt)  
- file2025 - the name of your ratings file for 2025 (from the causality project; start with this one: https://github.com/opendr-io/causality/blob/main/2025/September/2025-ratings-sep-14.txt)


In [452]:
# Define your vuln data file name here - note these are file names not paths.
vuln_path = 'test-data.csv' # Identify the vuln data file to ingest
# Provide the ratings files from the causality project https://github.com/opendr-io/causality
file2024 = 'reduction.txt'
file2025 = '2025-ratings-sep-14.txt'

In [363]:
# Load 2025 ratings and report results
with open(file2025, 'rb') as file:
    result = chardet.detect(file.read())
    encoding = result['encoding']

rating2025 = pd.read_csv(ratings2025, sep='\t', low_memory=False, encoding=encoding)

print("Unique value counts per field:\n")
for col in rating2025.columns:
    n_unique = rating2025[col].nunique(dropna=True)
    print(f"{col}: {n_unique} unique values")

all_nan = [col for col in rating2025.columns if rating2025[col].isna().all()]
if all_nan:
    print("\n⚠️ Fields entirely NaN:")
    for col in all_nan:
        print(f" - {col}")
else:
    print("\n✅ No fields are entirely NaN")

print("Shape of the 2025 ratings dataframe is:", rating2025.shape)


Unique value counts per field:

number: 6144 unique values
cve: 6144 unique values
assigner: 250 unique values
published: 6028 unique values
title: 3448 unique values
description: 5630 unique values
vendor: 1141 unique values
product: 2278 unique values
affected versions: 2367 unique values
rating: 2 unique values

✅ No fields are entirely NaN
Shape of the 2025 ratings dataframe is: (6144, 10)


In [364]:
# Load 2024 ratings and report results
with open(file2024, 'rb') as file:
    result = chardet.detect(file.read())
    encoding = result['encoding']
rating2024 = pd.read_csv(file2024, sep='\t', low_memory=False, encoding=encoding)

print("Unique value counts per field:\n")
for col in rating2024.columns:
    n_unique = rating2024[col].nunique(dropna=True)
    print(f"{col}: {n_unique} unique values")

all_nan = [col for col in rating2024.columns if rating2024[col].isna().all()]
if all_nan:
    print("\n⚠️ Fields entirely NaN:")
    for col in all_nan:
        print(f" - {col}")
else:
    print("\n✅ No fields are entirely NaN")

print("Shape of the 2024 ratings dataframe is:", rating2024.shape)

Unique value counts per field:

cve: 4238 unique values
vendorProject: 543 unique values
product: 1068 unique values
shortDescription: 3962 unique values

✅ No fields are entirely NaN
Shape of the 2024 ratings dataframe is: (4238, 4)


In [306]:
# Add EPSS scores
def get_all_epss_scores():
    """
    Download the complete EPSS dataset (all CVEs)
    This returns a large CSV file
    """
    url = "https://epss.cyentia.com/epss_scores-current.csv.gz"
    
    # Read compressed CSV - the file has a header row
    df = pd.read_csv(
        url, 
        compression='gzip',
        skiprows=1,  # Skip the first comment line
        dtype={
            'cve': str,
            'epss': float,
            'percentile': float
        },
        low_memory=False
    )
    return df

# Get all scores
df_epss = get_all_epss_scores()
df_epss.rename(columns={
    'epss': 'epss.score',
    'percentile': 'epss.percentile'
}, inplace=True)

print("Unique value counts per field:\n")
for col in df_epss.columns:
    n_unique = df_epss[col].nunique(dropna=True)
    print(f"{col}: {n_unique} unique values")

# Identify fields that are entirely NaN
all_nan = [col for col in df_epss.columns if df_epss[col].isna().all()]

if all_nan:
    print("\n⚠️ Fields entirely NaN:")
    for col in all_nan:
        print(f" - {col}")
else:
    print("\n✅ No fields are entirely NaN")

print("Shape of the epss dataframe is:", df_epss.shape)
print()
print(df_epss.head())
print(f"\nShape: {df_epss.shape}")
print(f"\nColumns: {df_epss.columns.tolist()}")

Unique value counts per field:

cve: 296426 unique values
epss.score: 24886 unique values
epss.percentile: 77595 unique values

✅ No fields are entirely NaN
Shape of the epss dataframe is: (296426, 3)

             cve  epss.score  epss.percentile
0  CVE-1999-0001     0.01076          0.77138
1  CVE-1999-0002     0.10737          0.93083
2  CVE-1999-0003     0.90362          0.99589
3  CVE-1999-0004     0.03215          0.86595
4  CVE-1999-0005     0.25334          0.96034

Shape: (296426, 3)

Columns: ['cve', 'epss.score', 'epss.percentile']


In [308]:
# Add KEV status
def get_latest_kev():
    """
    Download the latest KEV catalog from CISA
    """
    url = "https://www.cisa.gov/sites/default/files/csv/known_exploited_vulnerabilities.csv"
    
    kev = pd.read_csv(url, low_memory=False)
    
    # Make all column names lowercase
    kev.columns = kev.columns.str.lower()
    
    # Rename cveid to cve
    if 'cveid' in kev.columns:
        kev.rename(columns={'cveid': 'cve'}, inplace=True)
    
    return kev

kev = get_latest_kev()
print("KEV Data loaded successfully!")
print(f"Shape: {kev.shape}")
# Identify fields that are entirely NaN
all_nan = [col for col in kev.columns if kev[col].isna().all()]
if all_nan:
    print("\n⚠️ Fields entirely NaN:")
    for col in all_nan:
        print(f" - {col}")
else:
    print("\n✅ No fields are entirely NaN")
print(f"\nColumns: {kev.columns.tolist()}")
print()
print("Unique value counts per field:\n")
for col in kev.columns:
    n_unique = kev[col].nunique(dropna=True)
    print(f"{col}: {n_unique} unique values")

KEV Data loaded successfully!
Shape: (1427, 11)

✅ No fields are entirely NaN

Columns: ['cve', 'vendorproject', 'product', 'vulnerabilityname', 'dateadded', 'shortdescription', 'requiredaction', 'duedate', 'knownransomwarecampaignuse', 'notes', 'cwes']

Unique value counts per field:

cve: 1427 unique values
vendorproject: 235 unique values
product: 583 unique values
vulnerabilityname: 1122 unique values
dateadded: 335 unique values
shortdescription: 1343 unique values
requiredaction: 40 unique values
duedate: 355 unique values
knownransomwarecampaignuse: 2 unique values
notes: 1427 unique values
cwes: 210 unique values


In [498]:
import chardet, csv, pandas as pd, io, os, textwrap

path = vuln_path

# Detect encoding
with open(path, "rb") as f:
    raw = f.read()
det = chardet.detect(raw)
enc = det["encoding"] or "utf-8"

# Read header using csv.reader to count true fields (quotes respected)
first_newline_idx = raw.find(b"\n")
header_bytes = raw[: first_newline_idx if first_newline_idx != -1 else len(raw)]
header_text = header_bytes.decode(enc, errors="replace")
header_fields = next(csv.reader([header_text]))
expected_fields = len(header_fields)
expected_commas = expected_fields - 1

def count_commas_outside_quotes(line: str) -> int:
    """Counts commas that act as delimiters by scanning for double quotes.
    Handles doubled double-quotes "" as escaped quotes inside a quoted field."""
    count = 0
    i = 0
    in_quotes = False
    while i < len(line):
        ch = line[i]
        if ch == '"':
            if in_quotes:
                # If this is an escaped quote (""), skip the next one and stay in quotes
                if i + 1 < len(line) and line[i+1] == '"':
                    i += 2
                    continue
                else:
                    in_quotes = False
            else:
                in_quotes = True
        elif ch == ',' and not in_quotes:
            count += 1
        i += 1
    return count

offending_lines = []
with open(path, "r", encoding=enc, errors="replace", newline="") as f:
    for lineno, line in enumerate(f, start=1):
        if lineno == 1:
            continue
        commac = count_commas_outside_quotes(line.rstrip("\r\n"))
        if commac != expected_commas:
            offending_lines.append(lineno)

summary = {
    "Detected encoding": enc,
    "Fields in header": expected_fields,
    "Expected delimiter commas per line": expected_commas,
    "Total lines (including header)": raw.count(b"\n") + 1,
    "Offending line count (too few or too many commas)": len(offending_lines),
}

# Report results
if len(offending_lines) == 0:
    print("✅ No offending lines detected. File structure looks consistent.")
else:
    N = 25
    preview_df = pd.DataFrame({"offending_line_number": offending_lines[:N]})
    print("❌ First offending line numbers (up to 25)", preview_df)

    # Save full list for review
    full_list_path = "lines_with_wrong_comma_count.txt"
    with open(full_list_path, "w", encoding="utf-8") as out:
        out.write(f"# Header fields: {expected_fields} (expected commas per line: {expected_commas})\n")
        out.write("\n".join(map(str, offending_lines)))

    print(f"Full offending line list saved to {full_list_path}")

summary


✅ No offending lines detected. File structure looks consistent.


{'Detected encoding': 'Windows-1252',
 'Fields in header': 4,
 'Expected delimiter commas per line': 3,
 'Total lines (including header)': 200,
 'Offending line count (too few or too many commas)': 0}

In [484]:
# Ingest data from the file in vuln_path defined in cell 3 above. 
# Detect encoding and handle errors (encoding may vary) and do some quality checks
with open(vuln_path, 'rb') as file:
    result = chardet.detect(file.read())
    encoding = result['encoding']
print(f"Detected encoding: {encoding}")

if len(offending_lines) > 0:
    sys.exit(
        f"❌ DATA ERRORS! See previous cell which found {len(offending_lines)} offending lines in {vuln_path}.\n"
        f"First few bad lines: {offending_lines[:10]}"
    )

# Load with error handling in case the detected encoding has issues
try:
    vulns = pd.read_csv(vuln_path, low_memory=False, encoding=encoding, header=0)
    print(f"✓ Loaded successfully with {encoding} encoding")
except UnicodeDecodeError:
    print(f"⚠️ Failed with {encoding}, trying with error handling...")
    vulns = pd.read_csv(
        vuln_path, low_memory=False, encoding=encoding,
        encoding_errors='replace', header=0
    )
    print("✓ Loaded with character replacement")

vulns.columns = vulns.columns.str.strip().str.lower()

all_nan = [col for col in vulns.columns if vulns[col].isna().all()]
if all_nan:
    print("\n⚠️ Fields entirely NaN:")
    for col in all_nan:
        print(f" - {col}")
else:
    print("\n✅ No fields are entirely NaN")

print("Shape of the vulns dataframe is:", vulns.shape)
print()
print("Unique value counts per field:\n")
for col in vulns.columns:
    n_unique = vulns[col].nunique(dropna=True)
    print(f"{col}: {n_unique} unique values")

print()
print("Fields with NaN values in vulns:\n")
nan_counts = vulns.isna().sum()
total_rows = len(vulns)

cols_with_nan = nan_counts[nan_counts > 0]
if len(cols_with_nan) > 0:
    for col, count in cols_with_nan.items():
        nan_pct = (count / total_rows) * 100
        print(f"{col}: {count} NaN values ({nan_pct:.2f}%)")
else:
    print("✅ No columns have NaN values")

# Regex for CVE IDs
cve_pattern = re.compile(r"^CVE-\d{4}-\d+$")
# Rows where 'cve' is invalid
bad_cve_rows = vulns[~vulns['cve'].astype(str).str.match(cve_pattern)]

if not bad_cve_rows.empty:
    print()
    print(f"❌ Found {len(bad_cve_rows)} rows with invalid CVE values!")
    # Print the full row for each offending record
    for idx, row in bad_cve_rows.iterrows():
        print(f"\n--- Row {idx} ---")
        print(row.to_string())
else:
    print("✅ All values in 'cve' column look like valid CVEs")


Detected encoding: Windows-1252
✓ Loaded successfully with Windows-1252 encoding

✅ No fields are entirely NaN
Shape of the vulns dataframe is: (198, 4)

Unique value counts per field:

cve: 198 unique values
vendor: 77 unique values
product: 139 unique values
label: 3 unique values

Fields with NaN values in vulns:

vendor: 4 NaN values (2.02%)
product: 3 NaN values (1.52%)
✅ All values in 'cve' column look like valid CVEs


In [485]:
# Try to find columns containing 'severity' or 'status'

severity_cols = [col for col in vulns.columns if 'severity' in col.lower()]
status_cols = [col for col in vulns.columns if 'status' in col.lower()]

# Print unique values for severity columns
if severity_cols:
    print("=== SEVERITY FIELDS ===\n")
    for col in severity_cols:
        unique_vals = vulns[col].dropna().unique()
        print(f"{col}:")
        print(f"  Unique values: {unique_vals}")
        print(f"  Count: {len(unique_vals)}\n")
else:
    print("No fields containing 'severity' found\n")

# Print unique values for status columns
if status_cols:
    print("=== STATUS FIELDS ===\n")
    for col in status_cols:
        unique_vals = vulns[col].dropna().unique()
        print(f"{col}:")
        print(f"  Unique values: {unique_vals}")
        print(f"  Count: {len(unique_vals)}\n")
else:
    print("No fields containing 'status' found")

# Find columns containing 'cve' or 'vulnerability' (case insensitive)
matching_cols = [col for col in vulns.columns 
                 if 'cve' in col.lower() or 'vulnerability' in col.lower()]

if matching_cols:
    print(f"Found {len(matching_cols)} field(s) containing 'cve' or 'vulnerability':\n")
    for col in matching_cols:
        print(f"  - {col}")
else:
    print("No fields containing 'cve' or 'vulnerability' were found")

No fields containing 'severity' found

No fields containing 'status' found
Found 1 field(s) containing 'cve' or 'vulnerability':

  - cve


In [486]:
# Check out the field list above and identify your field that contains CVE IDs. Provide it to the function below
# so that we have normalized field names across dataframes.

SOURCE_CVE_FIELD = 'cveid'  # <-- change this as needed
colmap = {c.lower(): c for c in vulns.columns}
if SOURCE_CVE_FIELD.lower() in colmap:
    src = colmap[SOURCE_CVE_FIELD.lower()]
    if src == 'cve':
        pass  # already named 'cve'
    elif 'cve' in vulns.columns:
        print("Target column 'cve' already exists; skipping rename to avoid duplicate.")
    else:
        vulns.rename(columns={src: 'cve'}, inplace=True)
        print(f"Renamed '{SOURCE_CVE_FIELD}' to cve in order to play with the other dataframes.")
else:
    print(f"Column '{SOURCE_CVE_FIELD}' not found; nothing to rename. These could be CVE field names:")
    # Find columns containing 'cve' or 'vulnerability' (case insensitive)
    matching_cols = [col for col in vulns.columns 
                     if 'cve' in col.lower() or 'vulnerability' in col.lower()]
    
    if matching_cols:
        print(f"\nFound {len(matching_cols)} field(s) containing 'cve' or 'vulnerability':\n")
        for col in matching_cols:
            print(f"  - {col}")
    else:
        print("No fields containing 'cve' or 'vulnerability' were found")

Column 'cveid' not found; nothing to rename. These could be CVE field names:

Found 1 field(s) containing 'cve' or 'vulnerability':

  - cve


In [502]:
# Add or update a boolean 'kev' column to vulns indicating if its CVE appears in kev
if 'cve' not in vulns.columns:
    raise KeyError("vulns is missing 'cve' column")
if 'cve' not in kev.columns:
    raise KeyError("kev is missing 'cve' column")

# Check if 'kev' column already exists
if 'kev' in vulns.columns:
    print("'kev' column already exists; overwriting values...")
else:
    print("Creating 'kev' column...")

kev_lookup = set(
    kev['cve'].astype('string').str.strip().str.lower().dropna().unique()
)
vulns_cve_norm = vulns['cve'].astype('string').str.strip().str.lower()
vulns['kev'] = vulns_cve_norm.isin(kev_lookup).fillna(False)
vulns['kev'] = vulns['kev'].astype(bool)

total = len(vulns)
matches = int(vulns['kev'].sum())                  # count of True
populated = int(vulns['kev'].notna().sum())        # rows where 'kev' is not NaN
non_matches = populated - matches                  # or: int((~vulns['kev']).sum()
print(f"KEV matches (True): {matches:,}")
print(f"Non-matches (False): {non_matches:,}")
print(f"Rows populated in 'kev': {populated:,} (out of {total:,})")
# Consistency check
if populated == total:
    print("✅ All rows in 'kev' are populated")
else:
    print(f"❌ Mismatch: only {populated:,} of {total:,} rows are populated in 'kev'")


'kev' column already exists; overwriting values...
KEV matches (True): 2
Non-matches (False): 196
Rows populated in 'kev': 198 (out of 198)
✅ All rows in 'kev' are populated


In [500]:
# Re-extract year for ALL rows
vulns['year'] = vulns['cve'].astype(str).str.extract(r'CVE-(\d{4})-', expand=False)
vulns['year'] = pd.to_numeric(vulns['year'], errors='coerce').astype('Int64')

# Verify every row was processed
print(f"Total rows in vulns: {len(vulns):,}")
print(f"Rows with year value (not NaN): {vulns['year'].notna().sum():,}")
print(f"Rows with NaN year: {vulns['year'].isna().sum():,}")

# Print all unique values in the year field, including NaN
print("Unique values in 'year' field (including NaN):")
unique_years = vulns['year'].unique()
print(sorted([y for y in unique_years if pd.notna(y)]))

# Show if there are any NaN values
nan_count = vulns['year'].isna().sum()
if nan_count > 0:
    print(f"\nNaN values: {nan_count:,}")
    
# Set pandas to display all rows
pd.set_option('display.max_rows', None)

# Count all values in year field including NaN
print("Year value counts (including NaN):")
print(vulns['year'].value_counts(dropna=False).sort_index())


# Regex for a 4-digit year
year_pattern = re.compile(r"^\d{4}$")

# Make sure it's a string so regex can run safely
year_as_str = vulns['year'].astype(str)

# Rows where year is NaN, empty, or not matching 4-digit pattern
bad_year_rows = vulns[~year_as_str.str.match(year_pattern)]

if not bad_year_rows.empty:
    print(f"❌ Found {len(bad_year_rows)} rows without a valid year!")
    for idx, row in bad_year_rows.iterrows():
        print(f"\n--- Row {idx} ---")
        print(row.to_string())
else:
    print("✅ All rows have a valid year in the 'year' field")


# Reset display option if desired
# pd.reset_option('display.max_rows')

Total rows in vulns: 198
Rows with year value (not NaN): 198
Rows with NaN year: 0
Unique values in 'year' field (including NaN):
[2024, 2025]
Year value counts (including NaN):
year
2024    99
2025    99
<NA>     0
Name: count, dtype: Int64
✅ All rows have a valid year in the 'year' field


In [490]:
# 1. Ensure 'epss' field exists
if 'epss' not in vulns.columns:
    vulns['epss'] = np.nan

# 2. Build mapping from CVE → EPSS score
epss_map = df_epss.set_index('cve')['epss.score'].to_dict()

# 3. Update in place: only fill where match exists
vulns['epss'] = vulns['cve'].map(epss_map).combine_first(vulns['epss'])
total = len(vulns)

# Count rows with and without EPS score
with_epss = vulns['epss'].notna().sum()
without_epss = vulns['epss'].isna().sum()

# Percentages
pct_with = (with_epss / total) * 100
pct_without = (without_epss / total) * 100

print(f"Rows with EPS score: {with_epss} ({pct_with:.2f}%)")
print(f"Rows without EPS score: {without_epss} ({pct_without:.2f}%)")

if 'state' in vulns.columns:
    no_epss = vulns[vulns['epss'].isna()]
    assigner_counts = no_epss['state'].value_counts()
    print("Rows missing EPSS scores by state:")
    print(assigner_counts)
else:
    print("No 'state' field in vulns dataframe.")

#no_epss = vulns[vulns['epss'].isna()]
#no_epss.head(3)


Rows with EPS score: 196 (98.99%)
Rows without EPS score: 2 (1.01%)
No 'state' field in vulns dataframe.


In [ ]:
# Initialize the 'rating' field with 'none'
vulns['rating'] = 'none'
print("Initialized 'rating' field with 'none'")

# Verify required columns exist
if 'year' not in vulns.columns:
    raise KeyError("vulns is missing 'year' column")
if 'cve' not in vulns.columns:
    raise KeyError("vulns is missing 'cve' column")

# For 2025 ratings - use the actual rating value from rating2025
if 'cve' in rating2025.columns and 'rating' in rating2025.columns:
    # Create a lookup dictionary for 2025 ratings
    rating2025_dict = dict(zip(
        rating2025['cve'].astype('string').str.strip().str.upper(),
        rating2025['rating']
    ))
    
    # Update vulns for year 2025
    mask_2025 = vulns['year'] == 2025
    total_2025 = mask_2025.sum()
    
    if total_2025 > 0:
        vulns.loc[mask_2025, 'rating'] = vulns.loc[mask_2025, 'cve'].astype('string').str.strip().str.upper().map(rating2025_dict).fillna('cold')
        matches_2025 = (vulns.loc[mask_2025, 'rating'] != 'cold').sum()
        cold_2025 = (vulns.loc[mask_2025, 'rating'] == 'cold').sum()
        print(f"\n2025 ratings: {matches_2025:,} matched, {cold_2025:,} set to 'cold' out of {total_2025:,} CVEs ({(matches_2025/total_2025*100):.2f}% matched)")
    else:
        print(f"\n2025 ratings: No 2025 CVEs found in vulns")
else:
    print("\n⚠️ rating2025 is missing 'cve' or 'rating' column")

# For 2024 ratings - set to 'hot' if CVE appears in rating2024, otherwise 'cold'
if 'cve' in rating2024.columns:
    # Create a set of 2024 CVEs for fast lookup
    rating2024_cves = set(
        rating2024['cve'].astype('string').str.strip().str.upper().dropna().unique()
    )
    
    # Update vulns for year 2024 - set to 'hot' if CVE is in rating2024, 'cold' otherwise
    mask_2024 = vulns['year'] == 2024
    total_2024 = mask_2024.sum()
    
    if total_2024 > 0:
        vulns_cve_norm_2024 = vulns.loc[mask_2024, 'cve'].astype('string').str.strip().str.upper()
        vulns.loc[mask_2024, 'rating'] = vulns_cve_norm_2024.isin(rating2024_cves).map({True: 'hot', False: 'cold'})
        matches_2024 = (vulns.loc[mask_2024, 'rating'] == 'hot').sum()
        cold_2024 = (vulns.loc[mask_2024, 'rating'] == 'cold').sum()
        print(f"2024 ratings: {matches_2024:,} set to 'hot', {cold_2024:,} set to 'cold' out of {total_2024:,} CVEs ({(matches_2024/total_2024*100):.2f}% hot)")
    else:
        print(f"2024 ratings: No 2024 CVEs found in vulns")
else:
    print("\n⚠️ rating2024 is missing 'cve' column")

# Summary
print(f"\nRating distribution in vulns:")
print(vulns['rating'].value_counts())

# Group by year and rating, include NaN values
combo_counts = (
    vulns.groupby(['year', 'rating'], dropna=False)
         .size()
         .reset_index(name='count')
)

# Add percentage column
total = combo_counts['count'].sum()
combo_counts['percent'] = (combo_counts['count'] / total * 100).round(2)

# Rows where 'rating' is NaN or empty string
bad_rating_rows = vulns[vulns['rating'].isna() | (vulns['rating'].astype(str).str.strip() == "")]

if not bad_rating_rows.empty:
    print(f"❌ Found {len(bad_rating_rows)} rows with no rating!")
    for idx, row in bad_rating_rows.iterrows():
        print(f"\n--- Row {idx} ---")
        print(row.to_string())
else:
   print()
   print("✅ All rows have a value in the 'rating' field")
   print()
print(combo_counts)


In [497]:
# Filter for hot or warm ratings
hot_warm = vulns[vulns['rating'].isin(['hot', 'warm'])]

# Create filename with current date
current_date = datetime.now().strftime('%Y-%m-%d')
output_file = f'causality_output_{current_date}.csv'

# Export to CSV
hot_warm.to_csv(output_file, index=False, encoding='utf-8')

print(f"Exported {len(hot_warm):,} rows with 'hot' or 'warm' ratings to '{output_file}'")
print(f"\nBreakdown:")
print(hot_warm['rating'].value_counts())

Exported 88 rows with 'hot' or 'warm' ratings to 'causality_output_2025-10-03.csv'

Breakdown:
rating
hot     63
warm    25
Name: count, dtype: int64


The cell below is for optional troubleshooting if you have a a csv file with some unusual chars in it they can be hard to find.

In [ ]:
# For ingest troubleshooting
# - Detects encoding
# - Scans for problematic / non-ASCII characters (e.g., NBSP 0xA0, zero-width, smart quotes, dashes, BOM)
# - Reports counts and example line numbers for each issue
# - Flags lines with unbalanced double quotes (potential CSV breakage)
#
# You can tweak the `PROBLEM_CHARS` map to add/remove characters to check.

from collections import defaultdict, Counter
import chardet
import pandas as pd
import re

pd.set_option('display.max_rows', None)
vuln_path = "test-data.csv"

# Read as bytes and detect encoding
with open(vuln_path, "rb") as f:
    raw = f.read()

det = chardet.detect(raw)
encoding = det.get("encoding") or "utf-8"
confidence = det.get("confidence")

# Decode with 'replace' so we can see if any undecodable bytes exist (as �)
text = raw.decode(encoding, errors="replace")

# Define characters/symbols to check
PROBLEM_CHARS = {
    "NBSP (U+00A0)": "\u00A0",
    "Zero Width Space (U+200B)": "\u200B",
    "Zero Width No-Break Space / BOM (U+FEFF)": "\uFEFF",
    "Left Double Smart Quote (U+201C)": "\u201C",
    "Right Double Smart Quote (U+201D)": "\u201D",
    "Left Single Smart Quote (U+2018)": "\u2018",
    "Right Single Smart Quote (U+2019)": "\u2019",
    "En Dash (U+2013)": "\u2013",
    "Em Dash (U+2014)": "\u2014",
    "Ellipsis (U+2026)": "\u2026",
    "Minus Sign (U+2212)": "\u2212",
    "Non-ASCII Thin Space (U+2009)": "\u2009",
    "Replacement Char from decoding (U+FFFD)": "\uFFFD",
}

# Split into lines preserving line numbers (1-based)
lines = text.splitlines()

# Helper: find all occurrences with line numbers
char_occurrences = {name: [] for name in PROBLEM_CHARS}
any_non_ascii_occurs = []
unbalanced_quote_lines = []

for i, line in enumerate(lines, start=1):
    # Check specific characters
    for name, ch in PROBLEM_CHARS.items():
        if ch in line:
            char_occurrences[name].append(i)
    # Track any non-ASCII characters (exclude \t and normal space/newline)
    for ch in line:
        if ord(ch) > 126 and ch not in ("\t",):
            any_non_ascii_occurs.append((i, ch))
    # CSV sanity: check unbalanced double quotes (") after accounting for CSV escaping "" -> treat them as pairs
    # Count number of raw quote characters; consider "" as two quotes (which is fine). We just need even total.
    if line.count('"') % 2 != 0:
        unbalanced_quote_lines.append(i)

# Summaries
problem_rows_summary = []
for name, lines_list in char_occurrences.items():
    count = len(lines_list)
    if count > 0:
        example_lines = ", ".join(map(str, lines_list[:10]))
    else:
        example_lines = ""
    problem_rows_summary.append({"Character": name, "Occurrences (lines)": count, "Example line numbers": example_lines})

# Non-ASCII summary (top characters)
non_ascii_counter = Counter(ch for _, ch in any_non_ascii_occurs)
top_non_ascii = [{"Char": k, "Codepoint": f"U+{ord(k):04X}", "Count": v} for k, v in non_ascii_counter.most_common(20)]

# Build DataFrames for display
df_chars = pd.DataFrame(problem_rows_summary).sort_values("Occurrences (lines)", ascending=False)
df_non_ascii = pd.DataFrame(top_non_ascii)
df_misc = pd.DataFrame(
    [
        {"Item": "Detected encoding", "Value": encoding},
        {"Item": "Detection confidence", "Value": confidence},
        {"Item": "Total lines", "Value": len(lines)},
        {"Item": "Lines with unbalanced quotes", "Value": len(unbalanced_quote_lines)},
    ]
)

# Also return a concise textual summary
summary_output = {
    "encoding": encoding,
    "confidence": confidence,
    "lines_total": len(lines),
    "unbalanced_quote_lines_count": len(unbalanced_quote_lines),
    "problem_char_types_with_hits": [row["Character"] for row in problem_rows_summary if row["Occurrences (lines)"] > 0],
}
summary_output
print()
print("Problem characters by type (line counts & examples)", df_chars)
print()
print("Top non-ASCII characters found (first 20 by frequency)", df_non_ascii)
print()
print("File / parsing sanity summary", df_misc)

